In [ ]:
from dotenv import load_dotenv
from os import environ as env
from pathlib import Path
import numpy as np
import pandas as pd
from shared_utils.colors import okabe_ito as colors
load_dotenv("adm_full_script/vars.sh")
exp_path = Path(env.get('EXP_PATH'))
ref_path = Path(env.get('REF_PATH'))
# scores = pd.read_pickle(exp_path / "scores.pkl")
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import glob
# samples = os.listdir(blackhole_path / "images/IMAGENET128")
runs_path = glob.glob(str(ref_path / "ddim*full"))
for i, run in enumerate(runs_path):
    print(f"{i}: {run}") 

In [ ]:
# selected_runs = [1,2,3]
selected_runs = [0,2]
runs = [] 
for run_id in selected_runs:
    run_path = runs_path[run_id]
    run_name = run_path.split("/")[-1]
    data = {}
    data['scores'] = pd.read_pickle(Path(run_path) / "scores.pkl")
    data['path'] = run_path
    data['name'] = run_name
    runs.append(data)

In [ ]:
classifer_fid = 6.72588
classifier_precision = 0.72581449
classifier_recall = 0.62534

In [ ]:
colors

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size': 14,          # General font size
    'axes.titlesize': 16,     # Title size
    'axes.labelsize': 14,     # X and Y axis label size
    'xtick.labelsize': 12,    # X tick label size
    'ytick.labelsize': 12,    # Y tick label size
    'legend.fontsize': 14     # Legend text size
})

# Assuming your dataframe is already loaded as the variable `scores`
df = runs[0]['scores'].copy()
df_direct = runs[1]['scores'].copy()

# 1. Clean the Precision and Recall columns (extract numbers from brackets)
for col in ['Precision', 'Recall']:
    df[col] = df[col].apply(lambda x: x[0] if isinstance(x, list) else float(str(x).strip('[]')))
    df_direct[col] = df_direct[col].apply(lambda x: x[0] if isinstance(x, list) else float(str(x).strip('[]')))

# 2. Ensure data is sorted by N so the lines plot correctly from left to right
df = df.sort_values(by='N')
df_direct = df_direct.sort_values(by='N')

# 3. Define the color palette to match your previous formatting
sel_colors = {
    'Random': colors['blue'],
    'G.U.': colors['green'],
    'Realism': colors['yellow'],
    'Rarity': colors['purple'] 
}

# Set up the figure and subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Filtering Baseline Compared on FID, Precision, and Recall')

# Group the dataframe by the Method and plot each line
for method, group_df in df.groupby('Method'):
    # Standardize label name for G.U. to match "Ours" if preferred, or keep as is
    label_name = 'LLLA' if method == 'G.U.' else method
    color = sel_colors.get(method, 'black')
    
    ax1.plot(group_df['N'], group_df['FID'], marker='+', markersize=10, 
             linestyle='-', linewidth=2, label=label_name, color=color)
    
    ax2.plot(group_df['N'], group_df['Precision'], marker='+', markersize=10, 
             linestyle='-', linewidth=2, label=label_name, color=color)
    
    ax3.plot(group_df['N'], group_df['Recall'], marker='+', markersize=10, 
             linestyle='-', linewidth=2, label=label_name, color=color)


group_df_direct_gu = df_direct.query("Method == 'G.U.'")
    # Standardize label name for G.U. to match "Ours" if preferred, or keep as is
# color = colors.get("G.U.", 'black')
color = colors['orange'] 
    
ax1.plot(group_df_direct_gu['N'], group_df_direct_gu['FID'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"DNI", color=color)

ax2.plot(group_df_direct_gu['N'], group_df_direct_gu['Precision'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"DNI", color=color)

ax3.plot(group_df_direct_gu['N'], group_df_direct_gu['Recall'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"DNI", color=color)

# ax1.axhline(y=classifer_fid, color='black', linestyle=':', label='Classifier Guidance')
# ax2.axhline(y=classifier_precision, color='black', linestyle=':', label='Classifier Guidance')
# ax3.axhline(y=classifier_recall, color='black', linestyle=':', label='Classifier Guidance')

# Formatting Plot 1: FID
ax1.set_title('FID (↓)')
ax1.grid(True, linestyle='--', alpha=0.5)

# Formatting Plot 2: Precision
ax2.set_title('Precision (↑)')
ax2.grid(True, linestyle='--', alpha=0.5)

# Formatting Plot 3: Recall
ax3.set_title('Recall (↑)')
ax3.grid(True, linestyle='--', alpha=0.5)

# Apply shared labels, x-ticks, and a single bottom legend
unique_n_values = sorted(df['N'].unique())
for ax in [ax1, ax2, ax3]:
    ax.set_xticks(unique_n_values)
    ax.set_xlabel(f'Nr. of images kept after filtering (out of {df["N"].max()})')

# Extract legend handles from the first axis to create a shared figure legend
handles, labels = ax1.get_legend_handles_labels()

# Reorder the legend to match standard order (Random, Ours, Realism, Rarity)
order_map = {'Random': 0, 'Ours': 1, 'Realism': 2, 'Rarity': 3}
sorted_handles_labels = sorted(zip(handles, labels), key=lambda x: order_map.get(x[1], 4))
sorted_handles, sorted_labels = zip(*sorted_handles_labels)

fig.legend(sorted_handles, sorted_labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.05), frameon=True)

# Adjust layout to make room for the bottom legend
plt.tight_layout(rect=[0, 0.05, 1, 1])

# Save or show the plot
plt.savefig("../../figures/metrics_llla_dni.pdf", format="pdf", bbox_inches="tight")
plt.show()
# plt.savefig('figure_13_replicated.pdf', bbox_inches='tight') # Uncomment to save

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

plt.style.use('seaborn-v0_8-paper')
plt.rcParams.update({
    'font.size': 14,          # General font size
    'axes.titlesize': 16,     # Title size
    'axes.labelsize': 14,     # X and Y axis label size
    'xtick.labelsize': 12,    # X tick label size
    'ytick.labelsize': 12,    # Y tick label size
    'legend.fontsize': 12     # Legend text size
})

# Assuming your dataframe is already loaded as the variable `scores`
df = runs[0]['scores'].copy()
df_direct = runs[1]['scores'].copy()

# 1. Clean the Precision and Recall columns (extract numbers from brackets)
for col in ['Precision', 'Recall']:
    df[col] = df[col].apply(lambda x: x[0] if isinstance(x, list) else float(str(x).strip('[]')))
    df_direct[col] = df_direct[col].apply(lambda x: x[0] if isinstance(x, list) else float(str(x).strip('[]')))

# 2. Ensure data is sorted by N so the lines plot correctly from left to right
df = df.sort_values(by='N')
df_direct = df_direct.sort_values(by='N')

# 3. Define the color palette to match your previous formatting
colors = {
    'Random': colors['blue'],
    'G.U.': colors['green'],
    'Realism': 'tab:brown',
    'Rarity': colors['purple'] 
}

# Set up the figure and subplots
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Image generation results', fontsize=14)

group_df_gu = df.query("Method == 'G.U.'")
    # Standardize label name for G.U. to match "Ours" if preferred, or keep as is
# color = colors.get("G.U.", 'black')
color = "#DC3220"
    
ax1.plot(group_df_gu['N'], group_df_gu['FID'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"LLLA", color=color)

ax2.plot(group_df_gu['N'], group_df_gu['Precision'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"LLLA", color=color)

ax3.plot(group_df_gu['N'], group_df_gu['Recall'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"LLLA", color=color)


group_df_direct_gu = df_direct.query("Method == 'G.U.'")
    # Standardize label name for G.U. to match "Ours" if preferred, or keep as is
# color = colors.get("G.U.", 'black')
color = "#DC3220"
    
ax1.plot(group_df_direct_gu['N'], group_df_direct_gu['FID'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"DNI", color=color)

ax2.plot(group_df_direct_gu['N'], group_df_direct_gu['Precision'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"DNI", color=color)

ax3.plot(group_df_direct_gu['N'], group_df_direct_gu['Recall'], marker='+', markersize=10, 
            linestyle='--', linewidth=2, label=f"DNI", color=color)

ax1.axhline(y=classifer_fid, color='black', linestyle=':', label='Classifier Guidance')
ax2.axhline(y=classifier_precision, color='black', linestyle=':', label='Classifier Guidance')
ax3.axhline(y=classifier_recall, color='black', linestyle=':', label='Classifier Guidance')

# Formatting Plot 1: FID
ax1.set_title('FID (↓)')
ax1.grid(True, linestyle='--', alpha=0.5)

# Formatting Plot 2: Precision
ax2.set_title('Precision (↑)')
ax2.grid(True, linestyle='--', alpha=0.5)

# Formatting Plot 3: Recall
ax3.set_title('Recall (↑)')
ax3.grid(True, linestyle='--', alpha=0.5)

# Apply shared labels, x-ticks, and a single bottom legend
unique_n_values = sorted(df['N'].unique())
for ax in [ax1, ax2, ax3]:
    ax.set_xticks(unique_n_values)
    ax.set_xlabel(f'Nr. of images kept after filtering (out of {df["N"].max()})')

# Extract legend handles from the first axis to create a shared figure legend
handles, labels = ax1.get_legend_handles_labels()

# Reorder the legend to match standard order (Random, Ours, Realism, Rarity)
order_map = {'Random': 0, 'Ours': 1, 'Realism': 2, 'Rarity': 3}
sorted_handles_labels = sorted(zip(handles, labels), key=lambda x: order_map.get(x[1], 4))
sorted_handles, sorted_labels = zip(*sorted_handles_labels)

fig.legend(sorted_handles, sorted_labels, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.05), frameon=True)

# Adjust layout to make room for the bottom legend
plt.tight_layout(rect=[0, 0.05, 1, 1])

# Save or show the plot
plt.show()
plt.savefig("../../figures/metrics_llla_dni.pdf", format="pdf", bbox_inches="tight")
# plt.savefig('figure_13_replicated.pdf', bbox_inches='tight') # Uncomment to save